# Test fine-tuned Qwen3.5-4B on GDS generation

End-to-end pipeline: **prompt → glayout Python code → executed → GDS file → rendered PNG**.

Requires `finetune_qwen.ipynb` to have completed at least the LoRA save step (`qwen35_4b_gds_lora/lora_adapter/`).

## 1 · Config

In [12]:
import os, sys, torch
from pathlib import Path

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
os.environ.setdefault("PDK_ROOT", os.path.expanduser("~/pdks"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")

WORKSPACE   = Path.cwd().parent.parent
sys.path.insert(0, str(WORKSPACE / "src" / "gelochip"))

MODEL_ID     = "Qwen/Qwen3.5-4B"
ADAPTER_DIR  = Path.cwd() / "qwen35_4b_gds_lora" / "lora_adapter"
OUT_DIR      = Path.cwd() / "test_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LEN  = 4096
MAX_NEW_TOK  = 4096    # bumped — full glayout circuits need 3000+ tokens
TEMPERATURE  = 0.2
TOP_P        = 0.95
REPETITION_PENALTY = 1.0

print(f"Adapter   : {ADAPTER_DIR}  (exists={ADAPTER_DIR.exists()})")
print(f"Gen       : T={TEMPERATURE} top_p={TOP_P} rep_pen={REPETITION_PENALTY} max_tok={MAX_NEW_TOK}")

Adapter   : /home/irman/Gelochip/notebooks/sft_finetuning/qwen35_4b_gds_lora/lora_adapter  (exists=True)
Gen       : T=0.2 top_p=0.95 rep_pen=1.0 max_tok=4096


## 2 · Load fine-tuned model

In [13]:
from unsloth import FastVisionModel
import gc, bitsandbytes as bnb

# Same patches as the training notebook
import unsloth.models._utils as _u
_u._get_statistics = lambda *a, **k: None
_u.get_statistics  = lambda *a, **k: None
if not getattr(bnb.nn.Params4bit, "_patched_unsloth", False):
    _orig_new = bnb.nn.Params4bit.__new__
    def _patched_new(cls, *args, **kwargs):
        kwargs.pop("_is_hf_initialized", None)
        return _orig_new(cls, *args, **kwargs)
    bnb.nn.Params4bit.__new__ = _patched_new
    bnb.nn.Params4bit._patched_unsloth = True

gc.collect(); torch.cuda.empty_cache()

# Unsloth can load a LoRA adapter directory directly — it auto-detects the
# base model from adapter_config.json and applies the LoRA.
model_path = str(ADAPTER_DIR) if ADAPTER_DIR.exists() else MODEL_ID
print(f"Loading: {model_path}")

model, tokenizer = FastVisionModel.from_pretrained(
    model_path,
    load_in_4bit               = True,
    use_gradient_checkpointing = False,
    max_seq_length             = MAX_SEQ_LEN,
)

FastVisionModel.for_inference(model)
print(f"GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Tokenizer type    : {type(tokenizer).__name__}")

Loading: /home/irman/Gelochip/notebooks/sft_finetuning/qwen35_4b_gds_lora/lora_adapter
==((====))==  Unsloth 2026.5.6: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.616 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

GPU mem after load: 6.99 GB
Tokenizer type    : Qwen3VLProcessor


## 3 · Generation helpers

In [14]:
import re
from PIL import Image
from transformers import TextStreamer, StoppingCriteria, StoppingCriteriaList


class CodeBlockEndStopper(StoppingCriteria):
    """Stops the moment a closing ``` appears (after at least 10 tokens generated)."""
    def __init__(self, tokenizer, prompt_len: int):
        self.tokenizer  = tokenizer
        self.prompt_len = prompt_len
        self._last_len  = 0

    def __call__(self, input_ids, scores, **kw):
        gen_ids = input_ids[0, self.prompt_len:]
        if gen_ids.shape[0] - self._last_len < 10:
            return False
        self._last_len = gen_ids.shape[0]
        text = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        return "```" in text


def _run_model(chat_text: str, pil_image, max_new: int, stream: bool, use_code_stopper: bool) -> str:
    """Single forward pass — used by both plan and code passes."""
    if pil_image is not None:
        inputs = tokenizer(text=[chat_text], images=[pil_image],
                           return_tensors="pt", padding=True).to("cuda")
    else:
        inputs = tokenizer(text=[chat_text],
                           return_tensors="pt", padding=True).to("cuda")

    text_tok = getattr(tokenizer, "tokenizer", tokenizer)
    streamer = TextStreamer(text_tok, skip_prompt=True, skip_special_tokens=True) if stream else None

    prompt_len = inputs["input_ids"].shape[1]
    stopping = None
    if use_code_stopper:
        stopping = StoppingCriteriaList([CodeBlockEndStopper(text_tok, prompt_len)])

    with torch.inference_mode():
        out_ids = model.generate(
            **inputs,
            max_new_tokens     = max_new,
            temperature        = TEMPERATURE,
            top_p              = TOP_P,
            repetition_penalty = REPETITION_PENALTY,
            do_sample          = TEMPERATURE > 0,
            pad_token_id       = getattr(tokenizer, "eos_token_id", None)
                                 or getattr(text_tok, "eos_token_id", 0),
            streamer           = streamer,
            stopping_criteria  = stopping,
        )

    gen_ids = out_ids[0][prompt_len:]
    decoder = getattr(tokenizer, "decode", None) or text_tok.decode
    return decoder(gen_ids, skip_special_tokens=True).strip()


def _build_chat(messages, image_path: str | None) -> tuple[str, "Image.Image | None"]:
    pil_image = Image.open(image_path).convert("RGB") if image_path else None
    chat_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return chat_text, pil_image


def think(prompt: str, image_path: str | None = None, stream: bool = True,
          max_new: int = 512) -> str:
    """Pass 1 — short plan. Model thinks about the circuit; no code yet.
    Caps thinking at ~512 tokens so it can't run away."""
    msgs = [{"role": "user", "content":
             ([{"type": "image", "image": Image.open(image_path).convert('RGB')},
               {"type": "text",  "text": f"{prompt}\n\nThink briefly about which glayout primitives, ports, and routing this circuit needs. Output a SHORT plan (≤200 words). Do NOT write code."}]
              if image_path else
              [{"type": "text", "text":
                f"{prompt}\n\nThink briefly about which glayout primitives, ports, and routing this circuit needs. Output a SHORT plan (≤200 words). Do NOT write code."}])
            }]
    chat_text, pil = _build_chat(msgs, image_path)
    return _run_model(chat_text, pil, max_new=max_new, stream=stream, use_code_stopper=False)


def code_from_plan(prompt: str, plan: str, image_path: str | None = None,
                   stream: bool = True) -> str:
    """Pass 2 — write code given the plan. Assistant turn is prefilled with ```python\\n
    so the model is locked into code mode. Stops at closing ``` ."""
    msgs = [{"role": "user", "content":
             ([{"type": "image", "image": Image.open(image_path).convert('RGB')},
               {"type": "text",  "text":
                f"Task: {prompt}\n\nPlan:\n{plan}\n\nNow write the complete glayout Python code following the plan."}]
              if image_path else
              [{"type": "text", "text":
                f"Task: {prompt}\n\nPlan:\n{plan}\n\nNow write the complete glayout Python code following the plan."}])
            }]
    chat_text, pil = _build_chat(msgs, image_path)
    # Prefill: skip thinking, jump straight into a python code block
    chat_text += "<think>\n\n</think>\n\n```python\n"
    body = _run_model(chat_text, pil, max_new=MAX_NEW_TOK, stream=stream, use_code_stopper=True)
    return "```python\n" + body   # so extract_python finds the fence


def generate(prompt: str, image_path: str | None = None, stream: bool = True) -> dict:
    """Two-pass generation: think → code. Returns {'plan': ..., 'code_block': ...}."""
    print("━━━ PASS 1 / PLAN ━━━")
    plan = think(prompt, image_path=image_path, stream=stream)
    print("\n\n━━━ PASS 2 / CODE ━━━")
    code_block = code_from_plan(prompt, plan, image_path=image_path, stream=stream)
    print()
    return {"plan": plan, "code_block": code_block}


def extract_python(text: str) -> str | None:
    m = re.search(r"```(?:python|py)?\s*\n(.*?)```", text, re.DOTALL)
    if m:
        return m.group(1).strip()
    m = re.search(r"```(?:python|py)?\s*\n(.*)", text, re.DOTALL)
    return m.group(1).strip() if m else None


print("helpers ready: think(prompt) | code_from_plan(prompt, plan) | generate(prompt) → {plan, code_block}")

helpers ready: think(prompt) | code_from_plan(prompt, plan) | generate(prompt) → {plan, code_block}


## 4 · Execute generated code → GDS

In [15]:
import traceback

def run_code_to_gds(code: str, run_name: str) -> Path | None:
    """Execute the generated glayout code in a fresh namespace, return path to the new .gds."""
    work = OUT_DIR / run_name
    work.mkdir(parents=True, exist_ok=True)
    cwd = os.getcwd()
    os.chdir(work)
    try:
        ns = {"__name__": "__main__"}
        exec(compile(code, f"<{run_name}>", "exec"), ns)
    except Exception:
        print("--- Code raised an exception ---")
        traceback.print_exc()
        os.chdir(cwd)
        return None
    os.chdir(cwd)
    # find any GDS that was written
    gds_files = sorted(work.glob("*.gds"), key=lambda p: p.stat().st_mtime, reverse=True)
    return gds_files[0] if gds_files else None

## 5 · Render GDS → PNG

In [16]:
import klayout.lay as klay
from IPython.display import Image as IPyImage, display

def render_gds(gds_path: Path, width: int = 1200, height: int = 800) -> Path:
    png_path = gds_path.with_suffix(".png")
    lv = klay.LayoutView()
    lv.load_layout(str(gds_path), True)
    lv.max_hier()
    lv.zoom_fit()
    lv.save_image(str(png_path), width, height)
    return png_path

def show(path: Path):
    display(IPyImage(filename=str(path)))

## 6 · Full pipeline: prompt → GDS → PNG

In [17]:
def prompt_to_gds(prompt: str, run_name: str, image_path: str | None = None,
                  stream: bool = True) -> None:
    """Two-pass pipeline: plan (thinking) → code → exec → GDS + PNG."""
    print(f"━━━━━━━━━━━━━ {run_name} ━━━━━━━━━━━━━")
    print(f"PROMPT: {prompt}\n")

    result = generate(prompt, image_path=image_path, stream=stream)
    plan       = result["plan"]
    code_block = result["code_block"]
    code       = extract_python(code_block)

    run_dir = OUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "plan.txt").write_text(plan)
    (run_dir / "code_block.txt").write_text(code_block)

    if code is None:
        print("\n✗ Pass 2 produced no ```python``` block at all.")
        print(f"  Inspect: {run_dir}/code_block.txt")
        return

    (run_dir / "generated.py").write_text(code)
    print(f"\n━━━ EXTRACTED CODE ({len(code)} chars) ━━━")
    print(code[:600] + ("…" if len(code) > 600 else ""))
    print(f"\nSaved → {run_dir}/  (plan.txt, code_block.txt, generated.py)")

    print("\n━━━ EXECUTING generated.py ━━━")
    gds = run_code_to_gds(code, run_name)
    if gds:
        png = render_gds(gds)
        print(f"\n✓ GDS: {gds}\n✓ PNG: {png}\n")
        show(png)
    else:
        print("\n✗ Code ran but no .gds file was written.")

## 7 · Example prompts

Try the 15 circuit types the model was trained on:

In [18]:
prompt_to_gds(
    "Generate complete glayout Python code for a current mirror on gf180 PDK.",
    run_name="01_current_mirror",
)

━━━━━━━━━━━━━ 01_current_mirror ━━━━━━━━━━━━━
PROMPT: Generate complete glayout Python code for a current mirror on gf180 PDK.

━━━ PASS 1 / PLAN ━━━
Thinking Process:

1.  **Analyze the Request:**
    *   **Task:** Write glayout Python code for a current mirror.
    *   **PDK:** gf180 PDK.
    *   **Constraint:** Think briefly first. Output a SHORT plan (≤200 words). Do NOT write code in the plan.

2.  **Identify Necessary Components/Primitives:**
    *   Current mirror needs two transistors (reference and output).
    *   Needs gate, drain, and source ports.
    *   Needs a dummy load (resistor) for the reference branch.
   

KeyboardInterrupt: 

In [ ]:
prompt_to_gds(
    "Generate complete glayout Python code for a differential pair on gf180 PDK.",
    run_name="02_diff_pair",
)

In [ ]:
prompt_to_gds(
    "Generate complete glayout Python code for a two-stage operational amplifier on gf180 PDK.",
    run_name="03_opamp_twostage",
)

In [ ]:
# Test image → code (give it a reference layout, ask it to reproduce)
ref_image = WORKSPACE / "notebooks/datasets/current_mirror" / "current_mirror_preview.png"
if ref_image.exists():
    prompt_to_gds(
        "Identify this analog IC GDS layout and write the complete glayout Python code to reproduce it.",
        run_name="04_image_to_code",
        image_path=str(ref_image),
    )
else:
    print(f"reference image not found: {ref_image}")

## 8 · Your own prompt

Edit and rerun:

In [ ]:
MY_PROMPT = "Generate complete glayout Python code for a stacked NFET current mirror on gf180 PDK."
prompt_to_gds(MY_PROMPT, run_name="99_custom")